In [ ]:
from xgboost import XGBClassifier
import pandas as pd
import mlflow
import mlflow.xgboost
from sklearn.metrics import average_precision_score,classification_report
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# 1. Load dataset
df = pd.read_csv("dataset.csv")

# 2. Select features
X = df[
    [
        "type",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest"
    ]
].copy()

y = df["isFraud"]

# 3. Encode transaction type into numeric columns
X = pd.get_dummies(X, columns=["type"], dtype=int)

# 4. Keep all fraud records and sample normal records
fraud = df["isFraud"] == 1
normal_df = df.loc[~fraud].sample(
    n=100000, random_state=42
)

selected_indices = list(normal_df.index) + list(df.index[fraud])
X = X.loc[selected_indices]
y = y.loc[selected_indices]

# 5. Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
smote=SMOTE(random_state=42)
X_train_smote,y_train_smote=smote.fit_resample(X_train,y_train)
model=XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
mlflow.set_experiment("Fraud Detection")
mlflow.xgboost.autolog()

with mlflow.start_run():
    model.fit(X_train_smote,y_train_smote)
    y_prob=model.predict_proba(X_test)[:,1]
    pr_auc=average_precision_score(y_test,y_prob)
    mlflow.log_metric("test_pr_auc",pr_auc)

    print("Test PR-AUC:",pr_auc)
    print("MLflow tracking completed!")
model.fit(X_train_smote,y_train_smote)
#testing
from sklearn.metrics import(
    classification_report,
    confusion_matrix,
    average_precision_score
)

y_pred=model.predict(X_test)
y_prob=model.predict_proba(X_test)[:,1]
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print(average_precision_score(y_test,y_prob))

model.save_model("fraud_model.json")
print("Model Saved Successfully")

2026/10/24 15:14:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/24 15:14:40 INFO mlflow.store.db.utils: Updating database tables
2026/10/24 15:14:43 INFO mlflow.tracking.fluent: Experiment with name 'Fraud Detection' does not exist. Creating a new experiment.
2026/10/24 15:14:47 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\vaira\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare int

Test PR-AUC: 0.9953893952850571
MLflow tracking completed!


2026/10/24 15:15:02 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\vaira\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/10/24 15:15:02 WARNING mlflow.models.model: `artifact_pat

[[19877   123]
 [    8  1635]]
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     20000
           1       0.93      1.00      0.96      1643

    accuracy                           0.99     21643
   macro avg       0.96      0.99      0.98     21643
weighted avg       0.99      0.99      0.99     21643

0.9953893952850571
Model Saved Successfully
